# Структура обработанного датасета DRP-372 (только образцы 256x256x256)

## Папки

- data/npy/ - бинарные 3D-структуры пород (файлы *.npy)
- data/csv/ - целевые значения проницаемости (файлы *.csv)

## Формат .npy (входные данные)

- Размерность = (256, 256, 256)
- Тип данных: float32 (0 - поровое пространство, 1 - твёрдая фаза)
- Загрузка: `vol = np.load("data/npy/204_03_256.npy")`

## Формат .csv (целевые переменные)

Каждый CSV в data/csv/ содержит 6 строк (давления 1,2,5,10,20 МПа и LBM):

| pressure | permeability |
|----------|--------------|
| 1        |       |
| 2        |       |
| 5        |       |
| 10       |       |
| 20       |      |
| LBM      |     |

- pressure: давление в МПа (для LBM - строка 'LBM')
- permeability: проницаемость (при отсутствии данных - NaN)

Также все csv объединены в один файл с именем "dataset_combined.csv", расположенный в корне проекта

## PyTorch Dataset

Класс PermDataset возвращает пару (volume, target_vector):
- volume: тензор (1, 256, 256, 256)
- target: тензор (6,)

```python
dataset = PermDataset()
vol, perm = dataset[0]  # vol.shape=(1,256,256,256), perm.shape=(6,)

In [6]:
!pip install remotezip

Defaulting to user installation because normal site-packages is not writeable


## Получение и объединение .csv файлов с результатами симуляций, преобразование .mat -> .npy

In [2]:
from remotezip import RemoteZip
import h5py
import numpy as np
import pandas as pd
import os
import torch
from torch.utils.data import Dataset
import fnmatch
from tqdm.notebook import tqdm

url = "https://web.corral.tacc.utexas.edu/digitalporousmedia/archive/DRP-372/DRP-372_archive.zip"
os.makedirs("data/npy", exist_ok=True)
os.makedirs("data/csv", exist_ok=True)

def cb(name, obj):
  if isinstance(obj, h5py.Dataset) and obj.ndim == 3:
      return obj

def find_3d_dataset(h5_file):
  """Возвращает 3D массив или None."""
  ds = h5_file.visititems(cb)
  return ds

def get_permeability(csv_bytes):
  """Извлекает значение permeability из CSV."""
  try:
      text = csv_bytes.decode('utf-8')
      for line in text.splitlines():
          if 'permeability' in line.lower():
              parts = line.split(',')
              if len(parts) >= 2:
                  val = parts[1].strip()
                  if not val and len(parts) > 0:
                      val = parts[0].strip()
                  return float(val)
      return None
  except Exception as e:
      print(f"Ошибка парсинга: {e}")
      return None

with RemoteZip(url) as rz:
    all_files = rz.namelist()
    mat_files = [f for f in all_files if '_256.mat' in f and f.endswith('.mat')]

    for mat_path in tqdm(mat_files):
        name = mat_path.split('/')[-1][:-4]
        base_dir = '/'.join(mat_path.split('/')[:-1])

        #.mat -> .npy
        npy_path = f"data/npy/{name}.npy"
        if not os.path.exists(npy_path):
            try:
                rz.extract(mat_path, path='.')
                with h5py.File(mat_path, 'r') as hf:
                    vol = find_3d_dataset(hf)
                    if vol is not None:
                        np.save(npy_path, vol)
                os.remove(mat_path)
            except Exception as e:
                print(f"Ошибка при обработке {name}: {e}")
                continue

        # Поиск permeability
        perms = []
        pressures = [1,2,5,10,20,'LBM']
        for p in pressures:
            if p == 'LBM':
                pattern = f"{base_dir}/Single Phase*/LBM.csv"
            else:
                pattern = f"{base_dir}/Single Phase*/P_{p}_MPa.csv"
            matching = fnmatch.filter(all_files, pattern)
            if matching:
                csv_path = matching[0]
                try:
                    content = rz.read(csv_path)
                    perm = get_permeability(content)
                    perms.append(perm)
                except Exception as e:
                    print(f"Не удалось прочитать {csv_path}: {e}")
                    perms.append(None)
            else:
                perms.append(None)

        df_out = pd.DataFrame({'pressure': pressures, 'permeability': perms})
        df_out.to_csv(f"data/csv/{name}.csv", index=False)
        print(f"{name}: {perms}")




  0%|          | 0/136 [00:00<?, ?it/s]

317_02_256: [1.7531069478361092, 0.7624153232108687, 0.3672116813928819, 0.22045933940895926, 0.12627494639751016, 1.2627494639751017e-13]
374_02_03_256: [41.71824200766815, 23.39382825179733, 14.81054390724986, 10.79750869681798, 7.3604824660096755, 7.360482466009675e-12]
374_03_015_256: [11.077355113132537, 5.456470794585234, 2.813152775616743, 1.7521527300326838, 1.0502429365884796, 1.0502429365884795e-12]
374_09_02_256: [0.990834308981556, 0.44075643750433957, 0.19631521681851785, 0.10782334746938196, 0.05669857380200553, 5.669857380200552e-14]
374_07_02_256: [4.046186118928552, 1.8986294914880073, 0.9141169494090212, 0.5370872534526243, 0.30332613147969517, 3.0332613147969516e-13]
204_03_256: [0.2684027965832713, 0.12569862779265922, 0.04668785990902349, 0.02139093769971679, None, None]
65_03_256: [2.1348239560183573, 1.0288814472837888, 0.4769045668374405, 0.26819575440528665, 0.14353657116169724, 1.4353657116169723e-13]
10_01_256: [0.24533081259590386, 0.10793278113277016, 0.050

## Объединение .csv файлов в один и сохранение в корень

In [7]:
import pandas as pd
import os

csv_dir = "data/csv"
output_file = "dataset_combined.csv"

pressures = ['1','2','5','10','20','LBM']

rows = []
for idx, fname in enumerate(sorted(os.listdir(csv_dir)), start=1):
    if not fname.endswith(".csv"):
        continue
    sample_name = fname[:-4]
    df = pd.read_csv(os.path.join(csv_dir, fname))
    df['pressure'] = df['pressure'].astype(str)
    perm_dict = dict(zip(df['pressure'], df['permeability']))
    perms = [perm_dict.get(p, None) for p in pressures]
    perms = [float(x) if x is not None else None for x in perms]
    rows.append([idx, sample_name] + perms)

columns = ["index", "sample"] + [f"perm_{p}MPa" if p != 'LBM' else "perm_LBM" for p in pressures]
result_df = pd.DataFrame(rows, columns=columns)
result_df.to_csv(output_file, index=False)
print(f"Сохранён {output_file} с {len(result_df)} строками")


Сохранён dataset_combined.csv с 133 строками


## Создание класса датасета, унаследованного от torch.utils.data.Dataset

In [9]:
class PermeabilityDataset(Dataset):
    def __init__(self, npy_dir='data/npy', csv_dir='data/csv'):
        self.npy_dir = npy_dir
        self.csv_dir = csv_dir
        self.samples = [f[:-4] for f in os.listdir(npy_dir) if f.endswith('.npy')]
        self.samples = [s for s in self.samples if os.path.exists(f"{csv_dir}/{s}.csv")]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        vol = np.load(f"{self.npy_dir}/{name}.npy")
        vol_tensor = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)
        df = pd.read_csv(f"{self.csv_dir}/{name}.csv")
        perm = df['permeability'].values.astype(np.float32)
        perm = np.nan_to_num(perm, nan=0.0)
        return vol_tensor, torch.tensor(perm, dtype=torch.float32)


In [10]:
dataset = PermeabilityDataset()
print(f"Размер датасета: {len(dataset)}")
if len(dataset) > 0:
    sample_vol, sample_target = dataset[0]
    print(f"Пример volume: {sample_vol}")
    print(f"Пример target: {sample_target}")

Размер датасета: 133
Пример volume: tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         ...,

         [[0., 0., 0.,  ..., 1., 0., 0.],
          [0., 0., 0.,  ..., 1., 1., 0.],
          [0., 0., 0.,  ..., 1., 1., 0.],
          ...,
 